#### 1. Set up the environment

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve().parent

INSTALL_SCRIPT = PROJECT_DIR / "setup" / "install_requirements_t241.py"
REQUIREMENTS_FILE = PROJECT_DIR / "setup" / "requirements_t241.yml"

assert INSTALL_SCRIPT.exists(), INSTALL_SCRIPT
assert REQUIREMENTS_FILE.exists(), REQUIREMENTS_FILE

print("Project dir:", PROJECT_DIR)
print("Install script:", INSTALL_SCRIPT)
print("Requirements:", REQUIREMENTS_FILE)

!python "{INSTALL_SCRIPT}" --requirements "{REQUIREMENTS_FILE}" --project-dir "{PROJECT_DIR}"

Project dir: C:\Users\salat\Alessandro Salatiello - Alljoined
Install script: C:\Users\salat\Alessandro Salatiello - Alljoined\setup\install_requirements_t241.py
Requirements: C:\Users\salat\Alessandro Salatiello - Alljoined\setup\requirements_t241.yml

Project directory: C:\Users\salat\Alessandro Salatiello - Alljoined
Git repos directory: C:\Users\salat\Alessandro Salatiello - Alljoined\code

Python environment:
  executable: c:\Users\salat\anaconda3\envs\reve-allj\python.exe
  version:    3.11.15

No git section found. Skipping repo clone.

Initial torch status:
  version:        2.4.1+cu118
  cuda build:     True
  cuda version:   11.8
  cuda available: True
  gpu:            NVIDIA T500
  import path:    c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\torch\__init__.py

Torch already satisfies requirements. Skipping torch reinstall.

Installing regular packages:
  - numpy
  - scipy
  - matplotlib
  - pandas
  - scikit-learn
  - pyyaml
  - packaging
  - tqdm
  - ipywidget

#### 2. Preprocess data and train models

In [ ]:
# Imports modules
import importlib
import torch
import os
import utils

importlib.reload(utils)
from huggingface_hub import login

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
read_data, prep, dls, mods, trainers = utils.import_modules(
    ["read_data", "preprocessing", "dataloaders", "models", "trainers"]
)

# Set HF Token
hf_token = ""
if hf_token == "":
    raise ValueError("Please provide your HF token")
login(token=hf_token)

# Enable TRAINING mode to train the models (if disabled, the trained model checkpoints will be used)
TRAINING = True

# Enable hyperparameter optimization (if disabled, the default values in train_optuna.yml will be used)
HP_OPT = True

# Enable DEBUG mode to run the pipeline on a small data subset
DEBUG = True

# Subtrial mode
SUBTRIAL = True

# Set device, seed, and batch_size
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed = 42
batch_size = 32
subtrial_size = 1  # Number of seconds in one subtrial

# Read data, metadata, and configuration files
(
    data,
    labels,
    label2id,
    id2label,
    trial_info,
    channels,
    sf,
    configs,
    paths,
) = read_data.read(SUBTRIAL=SUBTRIAL)

# Main preprocessing and training loop
models = list(configs.keys())
# models = [models[0]]
metrics = {}
ypred_test = {}
for model in models:
    # Set random seed to ensure reproducibility
    print(f"\nHandling model '{model}' {'*' * 60}")
    utils.set_seeds(seed)

    # Run model-specific preprocessing pipeline
    print(f"Preprocessing {'*' * 68}")
    (
        data,
        data_spec,
        labels,
        channels,
        trials,
        n_subtrials_per_trial,
    ) = prep.preprocess(
        data=data,
        model_config=configs[model]["preprocessing"],
        sf=sf,
        DEBUG=DEBUG,
        SUBTRIAL=SUBTRIAL,
        labels=labels,
        channels=channels,
        subtrial_size=subtrial_size,
    )

    # Create dataloaders
    print(f"Creating trainin/validation/test dataloaders {'*' * 38}")
    dataloaders = dls.make_dataloaders(data, labels, batch_size, seed)

    # Train models (after hp optimization)
    if TRAINING:
        print(f"Training models{'*' * 68}")
        # print(f"Running hyperparameter optimization{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.optimize_model(
            model=model,
            dataloaders=dataloaders,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            HP_OPT=HP_OPT,
            device=device,
            home_folder=paths["results"],
        )

    else:
        print(f"Retrieving best trained model{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.retrieve_best_model(
            model=model,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            device=device,
            home_fld=paths["results"] / "models",
        )

    # Evaluate best model on test set
    print(f"Evaluating best model on test set{'*' * 38}")
    ypred_test[model] = trainers.evaluate_model(
        model=best_model,
        dataloader=dataloaders["test"],
        device=device,
        dummy_y=True,
    )["ypred"]

    # Convert subtrial predictions to trial predictions
    if SUBTRIAL:
        ypred_test[model] = prep.subtrials2trials(
            ypred_test[model],
            trials["test"],
            n_subtrials_per_trial["test"],
            modality="mode",
        )

    # Save best model predictions
    utils.save_preds(
        model,
        ypred_test[model],
        id2label,
        results_dir=paths["results"],
    )

    print(metrics[model])
    print(best_hyperparams)
    print()

# Print training summary and create summary plot
summary_df, plot_path = utils.summarize_model_metrics(
    metrics=metrics,
    dataloaders=dataloaders,
    n_classes=len(label2id),
)

C:\Users\salat\Alessandro Salatiello - Alljoined
Imported latest version of 'read_data' module
Imported latest version of 'preprocessing' module
Imported latest version of 'dataloaders' module
Imported latest version of 'models' module
Imported latest version of 'trainers' module

Handling model 'reve' ************************************************************
Preprocessing ********************************************************************
Filtering...
Extracting largest window...
Normalizing signals...
Applying zscore normalization...
Applying clip_std clipping...
Creating trainin/validation/test dataloaders **************************************
Training models********************************************************************


[I 2026-06-04 16:34:15,360] A new study created in memory with name: no-name-5a04bd81-eb52-4f91-b0e8-b110252a8589
[I 2026-06-04 16:34:18,902] Trial 0 finished with value: 0.4 and parameters: {'optimizer.lr': 8.262811077695198e-06, 'optimizer.weight_decay': 8.003270181962387e-06, 'scheduler.patience': 7}. Best is trial 0 with value: 0.4.
c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
[I 2026-06-04 16:34:21,209] Trial 1 finished with value: 0.4 and parameters: {'optimizer.lr': 0.0007746392817485381, 'optimizer.weight_decay': 5.105287831746945e-06, 'scheduler.patience': 8}. Best is trial 0 with value: 0.4.
c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\salat\anacond

Training model with best hyperparameters set...


Training:   0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-06-04 16:34:32,642] A new study created in memory with name: no-name-c4eeba14-eef7-48cd-bca0-cf5d3798e76a


...

Evaluating best model on test set**************************************
{'train': {'loss': 1.7835123538970947, 'accuracy': 0.2, 'balanced_accuracy': 0.16666666666666666, 'ypred': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}, 'valid': {'loss': 1.7920477390289307, 'accuracy': 0.2, 'balanced_accuracy': 0.2, 'ypred': array([1, 1, 1, 1, 1])}, 'best_epoch': 0, 'best_score': 0.2}
{'optimizer': {'lr': 0.0003792782880601087, 'weight_decay': 0.00037351003348570197}, 'scheduler': {'mode': 'max', 'factor': 0.1, 'min_lr': 1e-07, 'patience': 4}}


Handling model 'unishape' ************************************************************
Preprocessing ********************************************************************
Filtering...
Extracting largest window...
Normalizing signals...
Applying remapping normalization...
Creating trainin/validation/test dataloaders **************************************
Training models********************************************************************


[I 2026-06-04 16:34:46,673] Trial 0 finished with value: 0.2 and parameters: {'optimizer.lr': 2.3684307309854666e-05, 'optimizer.weight_decay': 0.0021102727105795586, 'scheduler.patience': 4}. Best is trial 0 with value: 0.2.
[I 2026-06-04 16:34:51,404] Trial 1 finished with value: 0.2 and parameters: {'optimizer.lr': 1.4642749711745458e-06, 'optimizer.weight_decay': 0.005890894821019972, 'scheduler.patience': 8}. Best is trial 0 with value: 0.2.
c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes

Training model with best hyperparameters set...


Training:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating best model on test set**************************************
{'train': {'loss': 1.8850150108337402, 'accuracy': 0.13333333333333333, 'balanced_accuracy': 0.16666666666666666, 'ypred': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}, 'valid': {'loss': 1.8937079906463623, 'accuracy': 0.2, 'balanced_accuracy': 0.2, 'ypred': array([0, 0, 0, 0, 0])}, 'best_epoch': 0, 'best_score': 0.2}
{'optimizer': {'lr': 2.3684307309854666e-05, 'weight_decay': 0.0021102727105795586}, 'scheduler': {'mode': 'max', 'factor': 0.1, 'min_lr': 1e-07, 'patience': 4}}


Model performance summary
                  model  train_accuracy  train_balanced_accuracy  valid_accuracy  valid_balanced_accuracy
                   reve          1.0000                   1.0000          0.4000                   0.4000
                cbramod          0.2000                   0.1667          0.2000                   0.2000
               unishape          0.1333                   0.1667          0.2000           